<a href="https://colab.research.google.com/github/prmatasari/Tugas-Besar-Fintech-MBA/blob/main/Tugas_Besar_MBA_Admission_dataset%2C_Class_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tugas Besar Keputusan Teknologi Financial-MBA Admission dataset, Class 2025**

### **1. Pendahuluan**
Proses seleksi calon mahasiswa pascasarjana, khususnya pada program Master of Business Administration (MBA), semakin mengandalkan analisis data untuk membantu pengambilan keputusan yang objektif dan akurat. Berbagai faktor seperti latar belakang akademik, pengalaman kerja, skor tes standar, serta karakteristik demografis sering menjadi indikator penting untuk menentukan apakah seorang pelamar layak diterima di suatu program MBA bergengsi.

Dataset yang digunakan dalam tugas besar ini adalah MBA Admission Dataset, Class 2025, yaitu dataset yang dikembangkan berdasarkan statistik penerimaan mahasiswa dari Wharton School. Dataset ini dirancang untuk keperluan edukasi terkait analisis data, eksplorasi hubungan antar variabel, dan pemodelan machine learning untuk memprediksi status penerimaan mahasiswa. Data set diambil dari link https://www.kaggle.com/datasets/taweilo/mba-admission-dataset/data.


Tujuan utama dari kajian ini adalah membangun model Machine Learning untuk memprediksi status penerimaan MBA berdasarkan fitur-fitur penting seperti:

*   GPA (Grade Point Average)
*   GMAT Score
*   Undergraduate Major
*   Work Experience
*   Work Industry
*   Demographic attributes (gender, race, international status)

Dengan melakukan eksplorasi data, pembersihan data, persiapan dataset, pemodelan, dan evaluasi, tugas ini diharapkan dapat memberikan pemahaman menyeluruh mengenai proses analisis data menggunakan Google Colab dan bagaimana model klasifikasi dapat membantu proses seleksi kandidat MBA di dunia nyata.

Pada penelitian ini, metode Machine Learning yang digunakan adalah model klasifikasi, seperti Logistic Regression dan Random Forest, yang dapat memprediksi status penerimaan dengan mengevaluasi pola dalam data berdasarkan variabel-variabel yang tersedia.



## **2. Data Loading and Initial Exploration**

Pada bagian ini dilakukan proses pemanggilan dataset ke dalam lingkungan Google Colab serta eksplorasi awal untuk memahami struktur dan kualitas data. Dataset yang digunakan adalah **MBA Admission Dataset, Class 2025** yang telah diunduh dalam bentuk file `MBA.csv`.

Langkah yang dilakukan meliputi:
1. Memuat dataset ke dalam sebuah DataFrame pandas.
2. Melihat ukuran dan beberapa baris pertama data.
3. Memeriksa tipe data tiap kolom.
4. Mengidentifikasi nilai hilang (missing values).
5. Melakukan pembersihan awal berdasarkan deskripsi dataset.

In [3]:
import pandas as pd
import numpy as np
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/MBA.csv')

# Preview data
df.head()

Mounted at /content/drive


,application_id,gender,international,gpa,major,race,gmat,work_exp,work_industry,admission
0,1,Female,False,3.30,Business,Asian,620.0,3.0,Financial Services,Admit
1,2,Male,False,3.28,Humanities,Black,680.0,5.0,Investment Management,NaN
2,3,Female,True,3.30,Business,NaN,710.0,5.0,Technology,Admit
3,4,Male,False,3.47,STEM,Black,690.0,6.0,Technology,NaN
4,5,Male,False,3.35,STEM,Hispanic,590.0,5.0,Consulting,NaN


Perintah `df.head()` digunakan untuk melihat beberapa baris pertama sebagai gambaran awal struktur data, termasuk nama kolom dan contoh nilai pada masing-masing atribut.

In [5]:
# Informasi tipe data setiap kolom
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6194 entries, 0 to 6193
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   application_id  6194 non-null   int64  
 1   gender          6194 non-null   object 
 2   international   6194 non-null   bool   
 3   gpa             6194 non-null   float64
 4   major           6194 non-null   object 
 5   race            4352 non-null   object 
 6   gmat            6194 non-null   float64
 7   work_exp        6194 non-null   float64
 8   work_industry   6194 non-null   object 
 9   admission       1000 non-null   object 
dtypes: bool(1), float64(3), int64(1), object(5)
memory usage: 441.7+ KB


Perintah `df.info()` menampilkan tipe data setiap kolom, jumlah nilai non-null,
serta apakah ada kolom yang masih bertipe object (kategorikal) atau numerik.
Informasi ini penting untuk menentukan jenis preprocessing dan pemodelan yang akan dilakukan pada tahap berikutnya.

In [6]:
# Jumlah nilai hilang di setiap kolom
df.isna().sum()

,0
application_id,0
gender,0
international,0
gpa,0
major,0
race,1842
gmat,0
work_exp,0
work_industry,0
admission,5194


Hasil `df.isna().sum()` menunjukkan jumlah nilai hilang (missing values) pada tiap kolom.
Berdasarkan dokumentasi dataset:
- Kolom **`admission`** menggunakan nilai `NaN` untuk menyatakan status *Deny*.
- Kolom **`race`** bernilai `NaN` untuk mahasiswa internasional.

Oleh karena itu, nilai hilang pada kolom tersebut tidak sepenuhnya error,
melainkan mengandung informasi yang harus dikonversi secara eksplisit pada langkah pembersihan data.

In [7]:
# 1) Mengubah NaN pada kolom admission menjadi kategori 'Deny'
df['admission'] = df['admission'].fillna('Deny')

# 2) Mengisi NaN pada kolom race sebagai 'International'
df['race'] = df['race'].fillna('International')

# Cek kembali apakah masih ada nilai hilang
df.isna().sum()

,0
application_id,0
gender,0
international,0
gpa,0
major,0
race,0
gmat,0
work_exp,0
work_industry,0
admission,0


Berdasarkan data card di Kaggle:
- Kolom **`admission`** memiliki tiga kategori: `Admit`, `Waitlist`, dan `Null` yang merepresentasikan `Deny`.
  Oleh karena itu, nilai `NaN` pada kolom ini diganti menjadi string `'Deny'` agar dapat digunakan sebagai kelas target dalam pemodelan klasifikasi.

- Kolom **`race`** bernilai `null` untuk mahasiswa internasional.
  Untuk menjaga konsistensi dan mempermudah analisis, nilai hilang pada kolom ini diisi dengan label `'International'`.

Setelah proses ini, seluruh kolom tidak lagi memiliki nilai hilang dan dataset siap digunakan untuk analisis statistik dan persiapan pemodelan pada bagian berikutnya.

In [8]:
print("Distribusi status admission:")
print(df['admission'].value_counts())

print("\nDistribusi gender:")
print(df['gender'].value_counts())

print("\nBeberapa nilai unik major:")
print(df['major'].value_counts())

Distribusi status admission:
admission
Deny        5194
Admit        900
Waitlist     100
Name: count, dtype: int64

Distribusi gender:
gender
Male      3943
Female    2251
Name: count, dtype: int64

Beberapa nilai unik major:
major
Humanities    2481
STEM          1875
Business      1838
Name: count, dtype: int64


Hasil eksplorasi awal menunjukkan distribusi kategori pada beberapa variabel penting dalam dataset.

Distribusi status penerimaan (**admission**) memperlihatkan bahwa sebagian besar pelamar berada pada kategori *Deny*, sementara jumlah pelamar yang berstatus *Admit* dan *Waitlist* relatif lebih sedikit. Hal ini mengindikasikan adanya ketidakseimbangan kelas (class imbalance) pada variabel target yang perlu diperhatikan pada tahap pemodelan selanjutnya.

Pada variabel **gender**, mayoritas pelamar berjenis kelamin *Male*, diikuti oleh *Female*. Distribusi ini memberikan gambaran demografis awal dari populasi pelamar dalam dataset.

Sementara itu, variabel **major** menunjukkan bahwa latar belakang pendidikan pelamar didominasi oleh bidang *Humanities*, diikuti oleh *STEM* dan *Business*. Informasi ini berguna untuk memahami komposisi akademik pelamar serta potensi pengaruhnya terhadap keputusan penerimaan MBA.

Temuan-temuan ini akan digunakan sebagai dasar untuk analisis statistik yang lebih mendalam serta persiapan dataset pada tahap selanjutnya.

## **3. Statistical Summary of the Dataset**

Pada bagian ini dilakukan analisis statistik deskriptif untuk memahami karakteristik numerik dari dataset MBA Admission, Class 2025. Analisis ini bertujuan untuk mengidentifikasi nilai pusat data, sebaran, serta potensi variasi pada masing-masing variabel numerik yang digunakan dalam pemodelan.

Statistik deskriptif yang dianalisis meliputi nilai minimum, maksimum, rata-rata (mean), standar deviasi, serta kuartil dari setiap variabel numerik. Hasil analisis ini digunakan sebagai dasar untuk memahami pola data dan menentukan langkah persiapan dataset pada tahap selanjutnya.

In [9]:
# Ringkasan statistik untuk variabel numerik
df.describe()

,application_id,gpa,gmat,work_exp
count,6194.000000,6194.000000,6194.000000,6194.000000
mean,3097.500000,3.250714,651.092993,5.016952
std,1788.198115,0.151541,49.294883,1.032432
min,1.000000,2.650000,570.000000,1.000000
25%,1549.250000,3.150000,610.000000,4.000000
50%,3097.500000,3.250000,650.000000,5.000000
75%,4645.750000,3.350000,680.000000,6.000000
max,6194.000000,3.770000,780.000000,9.000000


Hasil ringkasan statistik menunjukkan karakteristik utama dari variabel numerik dalam dataset MBA Admission.

- Variabel **GPA** memiliki nilai minimum sebesar **2.65** dan maksimum **3.77**, dengan nilai rata-rata **3.25**. Hal ini menunjukkan bahwa sebagian besar pelamar memiliki performa akademik yang relatif baik dan berada pada tingkat menengah hingga tinggi.
- Variabel **GMAT score** memiliki rentang nilai antara **570** hingga **780**, dengan nilai rata-rata sekitar **651**. Rentang nilai yang cukup lebar ini mengindikasikan adanya variasi kemampuan akademik dan kesiapan pelamar dalam mengikuti program MBA.
- Variabel **work experience (work_exp)** memiliki nilai minimum **1 tahun** dan maksimum **9 tahun**, dengan rata-rata sekitar **5 tahun** pengalaman kerja. Hal ini menunjukkan bahwa pelamar memiliki latar belakang profesional yang beragam, dari pelamar dengan pengalaman kerja terbatas hingga yang lebih berpengalaman.

Nilai standar deviasi pada masing-masing variabel menunjukkan adanya variasi data yang cukup signifikan, sehingga variabel-variabel numerik tersebut berpotensi memberikan kontribusi penting dalam membedakan status penerimaan MBA pada tahap pemodelan.


In [10]:
# Statistik numerik berdasarkan status admission
df.groupby('admission')[['gpa', 'gmat', 'work_exp']].mean()

,gpa,gmat,work_exp
admission,,,
Admit,3.354867,692.733333,5.046667
Deny,3.231457,643.444359,5.013862
Waitlist,3.313500,673.600000,4.910000


Analisis statistik berdasarkan status penerimaan menunjukkan adanya perbedaan nilai rata-rata pada beberapa variabel numerik utama.

Pelamar dengan status **Admit** memiliki nilai rata-rata **GPA (3.35)** dan **GMAT score (≈693)** yang lebih tinggi dibandingkan pelamar berstatus **Waitlist** maupun **Deny**. Hal ini mengindikasikan bahwa performa akademik memiliki peran penting dalam proses seleksi penerimaan MBA.

Selain itu, rata-rata **pengalaman kerja (work_exp)** pada kelompok *Admit* juga relatif lebih tinggi dibandingkan kelompok *Waitlist*, dan sebanding dengan kelompok *Deny*. Temuan ini menunjukkan bahwa pengalaman profesional turut menjadi faktor pertimbangan, meskipun pengaruhnya mungkin tidak sebesar faktor akademik.

Perbedaan nilai rata-rata antar kategori admission ini memberikan indikasi awal bahwa variabel-variabel numerik tersebut memiliki daya diskriminatif dan berpotensi dimanfaatkan secara efektif dalam pemodelan Machine Learning untuk memprediksi status penerimaan MBA.


In [11]:
# Statistik sederhana untuk variabel kategorikal
print("Distribusi admission:")
print(df['admission'].value_counts())

print("\nDistribusi international status:")
print(df['international'].value_counts())

Distribusi admission:
admission
Deny        5194
Admit        900
Waitlist     100
Name: count, dtype: int64

Distribusi international status:
international
False    4352
True     1842
Name: count, dtype: int64


Statistik sederhana pada variabel kategorikal memberikan gambaran mengenai komposisi kelas target dan karakteristik pelamar dalam dataset.

Distribusi variabel **admission** menunjukkan bahwa mayoritas pelamar berada pada kategori *Deny* (5.194 data), sedangkan jumlah pelamar dengan status *Admit* (900 data) dan *Waitlist* (100 data) jauh lebih sedikit. Kondisi ini menegaskan adanya ketidakseimbangan kelas (class imbalance) pada variabel target yang perlu diperhatikan pada tahap persiapan dataset dan pemodelan.

Pada variabel **international**, sebagian besar pelamar merupakan mahasiswa domestik (*False*), yaitu sebanyak 4.352 data, sementara mahasiswa internasional (*True*) berjumlah 1.842 data. Perbedaan proporsi ini menunjukkan adanya variasi karakteristik demografis yang berpotensi memengaruhi keputusan penerimaan dan dapat dimanfaatkan sebagai fitur dalam pemodelan Machine Learning.
